# P07 SkyOps — Week 6: Data Quality → Trusted Silver / Quarantine

### ZENAIZ × BVRIT Hyderabad Data Engineering Internship
**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Week:** 6 — Data Quality, Quarantine, Reconciliation and Replay  
**Technology:** Databricks + Spark SQL + Delta

## Week 6 objective

Start from the five Silver Candidate objects required by the approved Week 06 guide, apply the **eight governed DQ rules**, retain every applicable failure on the physical record, and route each record exactly once:

```text
Silver Candidate
      ↓
Reference validation + rule checks
      ↓
PASS ───────────────→ Trusted Silver
FAIL ───────────────→ Quarantine
      ↓
Candidate = Trusted + Quarantine
      ↓
Controlled correction + full replay
```

**Important:** Week 6 does not silently delete, repair, or overwrite failed business records. Gold must read Trusted Silver only.

## Source and implementation boundary

This notebook is generated from the supplied SkyOps Week 5 notebook, the Week 6 learning material, the project data dictionary / DQ requirements, and the approved Team 07 Week 06 DQ guide.

The approved guide requires:
- five batch Candidate objects: airports, carriers, routes, flights and delay components;
- all eight exact rule IDs;
- DQ metadata on routed records;
- independent evaluation with one physical quarantine row carrying all failed rule IDs;
- exact Flight and Cause reconciliation;
- a controlled correction/replay demonstration.

The supplied project files **do not state a numeric cause-reconciliation tolerance or an explicit numeric distance-band threshold table**. This notebook therefore exposes those as governed configuration cells instead of silently inventing business rules. The default cause tolerance is `0.0` for an exact reconciliation baseline; verify it against the approved playbook before final submission.

## 1. Week-5 handoff checkpoint

Week 5 produced these Candidate tables:

- `silver_airports_candidate`
- `silver_carriers_candidate`
- `silver_routes_candidate`
- `silver_flights_candidate`

Week 06 additionally requires `silver_candidate_delay_components`. Because the supplied source has five delay-cause fields inside `flights.csv` rather than a separate cause file, this notebook creates the cause Candidate at one **flight + cause type** physical grain from the Week-5 flight Candidate.

Run top-to-bottom, one executable cell at a time.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;

In [ ]:
%sql
SHOW TABLES;

In [ ]:
%sql
SELECT 'airports' AS entity, COUNT(*) AS candidate_rows
FROM silver_airports_candidate
UNION ALL
SELECT 'carriers', COUNT(*) FROM silver_carriers_candidate
UNION ALL
SELECT 'routes', COUNT(*) FROM silver_routes_candidate
UNION ALL
SELECT 'flights', COUNT(*) FROM silver_flights_candidate;

### Candidate handoff checkpoint

Do not continue if one of the four Week-5 Candidate tables is missing.

The Week 06 approved guide also requires a delay-component Candidate. The next cell derives it from the five documented cause columns without joining cause rows back to flights yet.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_delay_components
USING DELTA
AS
SELECT
    flight_id,
    source_record_key,
    'carrier_delay_minutes' AS cause_type,
    carrier_delay_minutes AS cause_minutes,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _bronze_schema_version,
    _bronze_record_hash,
    _rescued_payload,
    _candidate_created_at,
    _candidate_schema_version
FROM silver_flights_candidate
WHERE carrier_delay_minutes IS NOT NULL

UNION ALL
SELECT
    flight_id, source_record_key, 'weather_delay_minutes', weather_delay_minutes,
    _source_file_name, _source_file_path, _ingested_at, _ingestion_run_id,
    _bronze_schema_version, _bronze_record_hash, _rescued_payload,
    _candidate_created_at, _candidate_schema_version
FROM silver_flights_candidate
WHERE weather_delay_minutes IS NOT NULL

UNION ALL
SELECT
    flight_id, source_record_key, 'nas_delay_minutes', nas_delay_minutes,
    _source_file_name, _source_file_path, _ingested_at, _ingestion_run_id,
    _bronze_schema_version, _bronze_record_hash, _rescued_payload,
    _candidate_created_at, _candidate_schema_version
FROM silver_flights_candidate
WHERE nas_delay_minutes IS NOT NULL

UNION ALL
SELECT
    flight_id, source_record_key, 'security_delay_minutes', security_delay_minutes,
    _source_file_name, _source_file_path, _ingested_at, _ingestion_run_id,
    _bronze_schema_version, _bronze_record_hash, _rescued_payload,
    _candidate_created_at, _candidate_schema_version
FROM silver_flights_candidate
WHERE security_delay_minutes IS NOT NULL

UNION ALL
SELECT
    flight_id, source_record_key, 'late_aircraft_delay_minutes', late_aircraft_delay_minutes,
    _source_file_name, _source_file_path, _ingested_at, _ingestion_run_id,
    _bronze_schema_version, _bronze_record_hash, _rescued_payload,
    _candidate_created_at, _candidate_schema_version
FROM silver_flights_candidate
WHERE late_aircraft_delay_minutes IS NOT NULL;

In [ ]:
%sql
SELECT
    COUNT(*) AS cause_candidate_rows,
    COUNT(DISTINCT flight_id) AS distinct_flights,
    COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) AS distinct_flight_cause_keys
FROM silver_candidate_delay_components;

## 2. Governed Week 06 rulebook

The exact eight rules are:

| Rule | Severity | Applies to | Route |
|---|---|---|---|
| DQ-FLT-001 | CRITICAL | Flights | `quarantine_flights` |
| DQ-REF-001 | CRITICAL | Airports, carriers, routes, flights | entity-specific quarantine |
| DQ-TIM-001 | MAJOR | Flights | `quarantine_flights` |
| DQ-STS-001 | CRITICAL | Flights | `quarantine_flights` |
| DQ-DIV-001 | CRITICAL | Flights | `quarantine_flights` |
| DQ-DLY-001 | MAJOR | Flights | `quarantine_flights` |
| DQ-CAU-001 | MAJOR | Delay components + flights | `quarantine_delay_components` |
| DQ-RTE-001 | MAJOR | Routes + flights | `quarantine_routes` / `quarantine_flights` |

**Source boundary:** signed delay may be negative because a flight was early. Only operational delay must be nonnegative. Cancelled-flight actual values and unavailable elapsed/taxi values may legitimately be NULL. Cancellation code `U` is allowed for this source.

## 3. DQ configuration

### Cause tolerance

The supplied project documents require reconciliation "within the approved tolerance" but do not provide the numeric value. Use the default below only as a transparent execution setting and replace it if your approved playbook provides a different tolerance.

### Distance-band mapping

The supplied route reference contains the controlled `distance_band` values. The notebook validates a route's distance against a reference-derived band range rather than silently inventing a new business mapping. If the official playbook supplies fixed thresholds, replace the mapping cell with those thresholds.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW dq_config AS
SELECT
    CAST(0.0 AS DOUBLE) AS cause_reconciliation_tolerance;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW approved_distance_band_mapping AS
SELECT
    distance_band,
    MIN(distance_miles) AS approved_min_miles,
    MAX(distance_miles) AS approved_max_miles,
    COUNT(*) AS reference_rows
FROM silver_routes_candidate
WHERE distance_band IS NOT NULL
  AND distance_miles IS NOT NULL
GROUP BY distance_band;

SELECT * FROM approved_distance_band_mapping
ORDER BY approved_min_miles;

**Configuration checkpoint:** if the approved playbook supplies a formal tolerance or distance-band threshold mapping, use that governed value rather than the reference-derived fallback above.

## 4. Validate reference masters before flight checks

The approved sequence requires active carrier, airport and directed-route references to be validated before flight checks.

These helper views collapse the reference tables to one row per business reference. This avoids row multiplication when they are used by Flight DQ.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airport_reference_status AS
SELECT
    airport_code,
    SUM(CASE WHEN active_flag = 1 THEN 1 ELSE 0 END) AS active_rows,
    COUNT(*) AS total_rows
FROM silver_airports_candidate
GROUP BY airport_code;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carrier_reference_status AS
SELECT
    carrier_code,
    SUM(CASE WHEN active_flag = 1 THEN 1 ELSE 0 END) AS active_rows,
    COUNT(*) AS total_rows
FROM silver_carriers_candidate
GROUP BY carrier_code;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW route_reference_status AS
SELECT
    origin_airport_code,
    destination_airport_code,
    COUNT(*) AS route_rows,
    SUM(CASE WHEN route_id IS NOT NULL AND TRIM(route_id) <> '' THEN 1 ELSE 0 END) AS identified_route_rows,
    MIN(route_id) AS route_id,
    MIN(distance_miles) AS distance_miles,
    MIN(distance_band) AS distance_band
FROM silver_routes_candidate
GROUP BY origin_airport_code, destination_airport_code;

In [ ]:
%sql
SELECT 'airport' AS reference_type,
       SUM(CASE WHEN active_rows = 1 THEN 1 ELSE 0 END) AS valid_active_references,
       SUM(CASE WHEN active_rows <> 1 THEN 1 ELSE 0 END) AS invalid_references
FROM airport_reference_status
UNION ALL
SELECT 'carrier',
       SUM(CASE WHEN active_rows = 1 THEN 1 ELSE 0 END),
       SUM(CASE WHEN active_rows <> 1 THEN 1 ELSE 0 END)
FROM carrier_reference_status
UNION ALL
SELECT 'directed_route',
       SUM(CASE WHEN route_rows = 1 AND identified_route_rows = 1 THEN 1 ELSE 0 END),
       SUM(CASE WHEN route_rows <> 1 OR identified_route_rows <> 1 THEN 1 ELSE 0 END)
FROM route_reference_status;

## 5. DQ-REF-001 on reference Candidate tables

For airports and carriers, DQ-REF-001 is a master-validity check: the reference must resolve to exactly one active record.

For routes, the directed origin → destination pair must resolve to exactly one identified route.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airports_checked AS
SELECT
    a.*,
    CASE
      WHEN a.airport_code IS NULL OR TRIM(a.airport_code) = '' THEN 'FAIL'
      WHEN COALESCE(r.active_rows, 0) <> 1 THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_ref_001_check
FROM silver_airports_candidate a
LEFT JOIN airport_reference_status r
  ON a.airport_code = r.airport_code;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carriers_checked AS
SELECT
    c.*,
    CASE
      WHEN c.carrier_code IS NULL OR TRIM(c.carrier_code) = '' THEN 'FAIL'
      WHEN COALESCE(r.active_rows, 0) <> 1 THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_ref_001_check
FROM silver_carriers_candidate c
LEFT JOIN carrier_reference_status r
  ON c.carrier_code = r.carrier_code;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW routes_checked AS
SELECT
    r.*,

    CASE
      WHEN COALESCE(s.route_rows, 0) <> 1
        OR COALESCE(s.identified_route_rows, 0) <> 1
      THEN 'FAIL' ELSE 'PASS'
    END AS dq_ref_001_check,

    CASE
      WHEN r.origin_airport_code IS NULL
        OR r.destination_airport_code IS NULL
        OR TRIM(r.origin_airport_code) = ''
        OR TRIM(r.destination_airport_code) = ''
        OR r.origin_airport_code = r.destination_airport_code
        OR r.distance_miles IS NULL
        OR r.distance_miles <= 0
      THEN 'FAIL'
      WHEN d.route_id IS NOT NULL
           AND d.direction_count > 1
      THEN 'FAIL'
      WHEN b.approved_min_miles IS NULL
           OR r.distance_miles < b.approved_min_miles
           OR r.distance_miles > b.approved_max_miles
      THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_rte_001_check
FROM silver_routes_candidate r
LEFT JOIN route_reference_status s
  ON r.origin_airport_code = s.origin_airport_code
 AND r.destination_airport_code = s.destination_airport_code
LEFT JOIN (
    SELECT
        route_id,
        COUNT(DISTINCT CONCAT_WS('|', origin_airport_code, destination_airport_code)) AS direction_count
    FROM silver_routes_candidate
    WHERE route_id IS NOT NULL AND TRIM(route_id) <> ''
    GROUP BY route_id
) d
  ON r.route_id = d.route_id
LEFT JOIN approved_distance_band_mapping b
  ON r.distance_band = b.distance_band;

### Route-rule note

`DQ-RTE-001` requires positive distance, origin ≠ destination, consistent route identity/direction and distance-band agreement. The band check here uses the controlled route reference's observed range for each band because the supplied project files do not publish separate numeric thresholds.

## 6. DQ-FLT-001 — Flight identity and duplicate detection

Week 5 created `flight_id = TRIM(source_record_key)` and retained `source_record_key`. Week 6 now owns the business decision about identity and duplicate/conflicting records.

Do not choose a winner for a duplicate business key. Every physical Candidate row carrying the duplicate fails DQ-FLT-001.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_flight_ids AS
SELECT
    flight_id,
    COUNT(*) AS occurrences,
    COUNT(DISTINCT CONCAT_WS('|',
        COALESCE(reporting_carrier, ''),
        COALESCE(origin_airport_code, ''),
        COALESCE(destination_airport_code, ''),
        COALESCE(CAST(flight_number AS STRING), ''),
        COALESCE(CAST(flight_date AS STRING), '')
    )) AS distinct_business_values
FROM silver_flights_candidate
WHERE flight_id IS NOT NULL AND TRIM(flight_id) <> ''
GROUP BY flight_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
SELECT *
FROM duplicate_flight_ids
ORDER BY occurrences DESC
LIMIT 20;

## 7. DQ-TIM-001 — HHMM and supplied duration checks

Allowed source NULLs are preserved. A NULL actual time or unavailable elapsed/taxi measure is not automatically a failure.

HHMM validity:
- hour must be 0–23;
- minute must be 0–59.

Supplied elapsed/taxi durations must be nonnegative and within reasonable source bounds. This notebook uses the explicit nonnegative requirement from the rule and avoids inventing a tighter upper bound.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_time_checked AS
SELECT
    f.*,
    CASE
      WHEN scheduled_departure_hhmm IS NULL THEN 'FAIL'
      WHEN FLOOR(scheduled_departure_hhmm / 100) >= 24
        OR MOD(scheduled_departure_hhmm, 100) >= 60
      THEN 'FAIL'
      WHEN actual_departure_hhmm IS NOT NULL
       AND (FLOOR(actual_departure_hhmm / 100) >= 24
         OR MOD(actual_departure_hhmm, 100) >= 60)
      THEN 'FAIL'
      WHEN scheduled_arrival_hhmm IS NULL THEN 'FAIL'
      WHEN FLOOR(scheduled_arrival_hhmm / 100) >= 24
        OR MOD(scheduled_arrival_hhmm, 100) >= 60
      THEN 'FAIL'
      WHEN actual_arrival_hhmm IS NOT NULL
       AND (FLOOR(actual_arrival_hhmm / 100) >= 24
         OR MOD(actual_arrival_hhmm, 100) >= 60)
      THEN 'FAIL'
      WHEN scheduled_elapsed_minutes IS NOT NULL AND scheduled_elapsed_minutes < 0 THEN 'FAIL'
      WHEN actual_elapsed_minutes IS NOT NULL AND actual_elapsed_minutes < 0 THEN 'FAIL'
      WHEN air_time_minutes IS NOT NULL AND air_time_minutes < 0 THEN 'FAIL'
      WHEN taxi_out_minutes IS NOT NULL AND taxi_out_minutes < 0 THEN 'FAIL'
      WHEN taxi_in_minutes IS NOT NULL AND taxi_in_minutes < 0 THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_tim_001_check
FROM silver_flights_candidate f;

## 8. DQ-STS-001 — Cancellation status

Allowed cancellation codes are `A`, `B`, `C`, `D`, and source-limitation code `U`.

Cancelled flights must not carry trusted completed-flight arrival/elapsed measures. The source may legitimately leave unavailable actual values NULL.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_status_checked AS
SELECT
    f.*,
    CASE
      WHEN cancelled_flag NOT IN (0,1) THEN 'FAIL'
      WHEN cancelled_flag = 1
       AND (cancellation_code IS NULL
         OR cancellation_code NOT IN ('A','B','C','D','U'))
      THEN 'FAIL'
      WHEN cancelled_flag = 1
       AND (
            actual_arrival_hhmm IS NOT NULL
         OR arrival_delay_signed_minutes IS NOT NULL
         OR arrival_delay_minutes IS NOT NULL
         OR actual_elapsed_minutes IS NOT NULL
         OR air_time_minutes IS NOT NULL
         OR taxi_out_minutes IS NOT NULL
         OR taxi_in_minutes IS NOT NULL
       )
      THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_sts_001_check
FROM flights_time_checked f;

## 9. DQ-DIV-001 — Diversion consistency

The rule requires diversion status and fields to be logically compatible, and a record must not be simultaneously represented as an ordinary completion and cancellation/diversion.

The source dictionary provides `diverted_flag` but no separate diversion destination field, so the check uses the fields actually supplied by the project.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_diversion_checked AS
SELECT
    f.*,
    CASE
      WHEN diverted_flag NOT IN (0,1) THEN 'FAIL'
      WHEN diverted_flag = 1 AND cancelled_flag = 1 THEN 'FAIL'
      WHEN diverted_flag = 1
       AND actual_arrival_hhmm IS NOT NULL
       AND arrival_delay_minutes IS NOT NULL
       AND actual_elapsed_minutes IS NOT NULL
       AND cancelled_flag = 0
       AND actual_arrival_hhmm = scheduled_arrival_hhmm
      THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_div_001_check
FROM flights_status_checked f;

## 10. DQ-DLY-001 — Delay semantics

The project explicitly distinguishes signed source delay from operational delay:

- signed delay may be negative because a flight was early;
- operational delay must be nonnegative;
- operational delay must equal `max(signed delay, 0)`.

The supplied Week-5 schema does not contain a separate source 15-minute delayed flag. Therefore the notebook derives the governed 15-minute interpretation from the signed delay and documents that no independent source flag exists to compare against.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_delay_checked AS
SELECT
    f.*,

    GREATEST(COALESCE(departure_delay_signed_minutes, 0D), 0D)
        AS expected_departure_operational_delay,

    CASE
      WHEN departure_delay_minutes IS NULL
       AND departure_delay_signed_minutes IS NULL
      THEN 'PASS'
      WHEN departure_delay_minutes IS NULL
      THEN 'FAIL'
      WHEN departure_delay_minutes < 0
      THEN 'FAIL'
      WHEN departure_delay_minutes
           <> GREATEST(COALESCE(departure_delay_signed_minutes, 0D), 0D)
      THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_dly_001_check,

    CASE
      WHEN departure_delay_signed_minutes IS NULL THEN NULL
      WHEN departure_delay_signed_minutes >= 15 THEN 1
      ELSE 0
    END AS derived_delayed_15_flag
FROM flights_diversion_checked f;

### DQ-DLY-001 diagnostic

If your project team has a separately governed 15-minute flag in another approved Candidate field, add its comparison here. Do not invent a second source flag when the supplied data dictionary does not contain one.

In [ ]:
%sql
SELECT
    COUNT(*) AS candidate_rows,
    SUM(CASE WHEN dq_dly_001_check = 'FAIL' THEN 1 ELSE 0 END) AS delay_rule_failures,
    SUM(CASE WHEN departure_delay_signed_minutes < 0 THEN 1 ELSE 0 END) AS early_flights_allowed
FROM flights_delay_checked;

## 11. DQ-RTE-001 on Flights

A Flight fails route/distance validation when:
- origin equals destination;
- its directed origin → destination reference is duplicated/inconsistent;
- distance is non-positive;
- the flight distance does not agree with the approved directed route reference or its band mapping.

Reference validation is performed through one-row helper views to avoid multiplying Flight records.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_route_checked AS
SELECT
    f.*,
    CASE
      WHEN f.origin_airport_code IS NULL
        OR f.destination_airport_code IS NULL
        OR f.origin_airport_code = f.destination_airport_code
      THEN 'FAIL'
      WHEN f.distance_miles IS NULL OR f.distance_miles <= 0
      THEN 'FAIL'
      WHEN COALESCE(r.route_rows, 0) <> 1
      THEN 'FAIL'
      WHEN r.distance_miles IS NULL
        OR ABS(f.distance_miles - r.distance_miles) > 0
      THEN 'FAIL'
      WHEN b.approved_min_miles IS NULL
        OR f.distance_miles < b.approved_min_miles
        OR f.distance_miles > b.approved_max_miles
      THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_rte_001_check
FROM flights_delay_checked f
LEFT JOIN route_reference_status r
  ON f.origin_airport_code = r.origin_airport_code
 AND f.destination_airport_code = r.destination_airport_code
LEFT JOIN approved_distance_band_mapping b
  ON r.distance_band = b.distance_band;

## 12. DQ-REF-001 on Flights

A Flight must resolve to:
- exactly one active reporting carrier;
- exactly one active origin airport;
- exactly one active destination airport;
- exactly one approved directed route.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_reference_checked AS
SELECT
    f.*,
    CASE
      WHEN COALESCE(c.active_rows, 0) <> 1
        OR COALESCE(o.active_rows, 0) <> 1
        OR COALESCE(d.active_rows, 0) <> 1
        OR COALESCE(r.route_rows, 0) <> 1
      THEN 'FAIL'
      ELSE 'PASS'
    END AS dq_ref_001_check
FROM flights_route_checked f
LEFT JOIN carrier_reference_status c
  ON f.reporting_carrier = c.carrier_code
LEFT JOIN airport_reference_status o
  ON f.origin_airport_code = o.airport_code
LEFT JOIN airport_reference_status d
  ON f.destination_airport_code = d.airport_code
LEFT JOIN route_reference_status r
  ON f.origin_airport_code = r.origin_airport_code
 AND f.destination_airport_code = r.destination_airport_code;

## 13. Combine all Flight checks — no short-circuit logic

Every applicable rule is evaluated independently before routing.

A single physical Flight can fail multiple rules. The quarantine table must still contain that physical record only once.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_all_checked AS
SELECT
    f.*,

    CASE
      WHEN f.flight_id IS NULL OR TRIM(f.flight_id) = ''
        OR d.flight_id IS NOT NULL
      THEN 'FAIL' ELSE 'PASS'
    END AS dq_flt_001_check,

    f.dq_ref_001_check,
    f.dq_tim_001_check,
    f.dq_sts_001_check,
    f.dq_div_001_check,
    f.dq_dly_001_check,
    f.dq_rte_001_check

FROM flights_reference_checked f
LEFT JOIN duplicate_flight_ids d
  ON f.flight_id = d.flight_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_dq AS
SELECT
    *,
    filter(array(
      CASE WHEN dq_flt_001_check = 'FAIL' THEN 'DQ-FLT-001' END,
      CASE WHEN dq_ref_001_check = 'FAIL' THEN 'DQ-REF-001' END,
      CASE WHEN dq_tim_001_check = 'FAIL' THEN 'DQ-TIM-001' END,
      CASE WHEN dq_sts_001_check = 'FAIL' THEN 'DQ-STS-001' END,
      CASE WHEN dq_div_001_check = 'FAIL' THEN 'DQ-DIV-001' END,
      CASE WHEN dq_dly_001_check = 'FAIL' THEN 'DQ-DLY-001' END,
      CASE WHEN dq_rte_001_check = 'FAIL' THEN 'DQ-RTE-001' END
    ), x -> x IS NOT NULL) AS failed_rule_ids,

    filter(array(
      CASE WHEN dq_flt_001_check = 'FAIL' THEN 'Flight identity missing, duplicated or conflicting' END,
      CASE WHEN dq_ref_001_check = 'FAIL' THEN 'Carrier, airport or directed route reference is not exactly one active approved reference' END,
      CASE WHEN dq_tim_001_check = 'FAIL' THEN 'HHMM or supplied duration is invalid' END,
      CASE WHEN dq_sts_001_check = 'FAIL' THEN 'Cancellation status/code/completed-flight measures are incompatible' END,
      CASE WHEN dq_div_001_check = 'FAIL' THEN 'Diversion status is incompatible with completion/cancellation' END,
      CASE WHEN dq_dly_001_check = 'FAIL' THEN 'Operational delay is inconsistent with signed source delay' END,
      CASE WHEN dq_rte_001_check = 'FAIL' THEN 'Route origin/destination, distance or distance-band validation failed' END
    ), x -> x IS NOT NULL) AS failure_reasons,

    array_join(
      filter(array(
        CASE WHEN dq_flt_001_check = 'FAIL' THEN 'flight_id/source_record_key' END,
        CASE WHEN dq_ref_001_check = 'FAIL' THEN 'reporting_carrier/origin/destination/route' END,
        CASE WHEN dq_tim_001_check = 'FAIL' THEN 'HHMM/duration fields' END,
        CASE WHEN dq_sts_001_check = 'FAIL' THEN 'cancelled_flag/cancellation_code/completed measures' END,
        CASE WHEN dq_div_001_check = 'FAIL' THEN 'diverted_flag/status fields' END,
        CASE WHEN dq_dly_001_check = 'FAIL' THEN 'signed/operational delay' END,
        CASE WHEN dq_rte_001_check = 'FAIL' THEN 'origin/destination/distance/route' END
      ), x -> x IS NOT NULL), '; ') AS affected_fields
FROM flights_all_checked;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_routed AS
SELECT
    *,
    CASE WHEN size(failed_rule_ids) = 0 THEN 'PASS' ELSE 'FAIL' END AS dq_status,
    CASE
      WHEN array_contains(failed_rule_ids, 'DQ-FLT-001')
        OR array_contains(failed_rule_ids, 'DQ-REF-001')
        OR array_contains(failed_rule_ids, 'DQ-STS-001')
        OR array_contains(failed_rule_ids, 'DQ-DIV-001')
      THEN 'CRITICAL'
      WHEN size(failed_rule_ids) > 0 THEN 'MAJOR'
      ELSE 'NONE'
    END AS severity,
    flight_id AS physical_record_key,
    _source_file_name AS source_file,
    _ingestion_run_id AS batch_id,
    CASE WHEN size(failed_rule_ids) > 0 THEN current_timestamp() ELSE NULL END AS quarantined_at,
    CAST(NULL AS STRING) AS replay_run_id
FROM flights_dq;

In [ ]:
%sql
SELECT
    dq_status,
    severity,
    COUNT(*) AS rows
FROM flights_routed
GROUP BY dq_status, severity
ORDER BY dq_status, severity;

## 14. DQ-CAU-001 — Delay-cause reconciliation

Cause rows are evaluated at their own physical grain. They are **aggregated before being joined back to flight metrics**, preventing one flight from multiplying into several KPI rows.

The rule fails when:
- a cause minute is negative;
- cause rows exist for an ineligible/not-delayed flight;
- cause minutes do not reconcile to operational delay within the governed tolerance.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW cause_aggregates AS
SELECT
    flight_id,
    source_record_key,
    SUM(cause_minutes) AS cause_total_minutes,
    MIN(cause_minutes) AS minimum_cause_minutes,
    COUNT(*) AS cause_row_count
FROM silver_candidate_delay_components
GROUP BY flight_id, source_record_key;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW causes_checked AS
SELECT
    c.*,
    f.departure_delay_signed_minutes,
    f.departure_delay_minutes AS operational_delay_minutes,
    GREATEST(COALESCE(f.departure_delay_signed_minutes, 0D), 0D)
        AS expected_operational_delay,
    CASE
      WHEN c.cause_minutes < 0 THEN 'FAIL'
      ELSE 'PASS'
    END AS cause_negative_check,
    CASE
      WHEN c.cause_minutes IS NOT NULL
       AND (f.flight_id IS NULL OR f.departure_delay_minutes < 15)
      THEN 'FAIL'
      ELSE 'PASS'
    END AS cause_eligibility_check
FROM silver_candidate_delay_components c
LEFT JOIN silver_flights_candidate f
  ON c.flight_id = f.flight_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW causes_reconciled AS
SELECT
    a.*,
    CASE
      WHEN a.cause_total_minutes IS NULL THEN 'PASS'
      WHEN ABS(a.cause_total_minutes - a.expected_operational_delay)
             > (SELECT cause_reconciliation_tolerance FROM dq_config)
      THEN 'FAIL'
      ELSE 'PASS'
    END AS cause_reconciliation_check
FROM cause_aggregates a;

### Cause grain and reconciliation note

The source files do not publish a separate delay-component file. The Candidate cause rows are therefore derived from the five cause columns in the flight Candidate. The physical cause key is `flight_id + cause_type`; the reconciliation is performed after aggregation to `flight_id`.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW causes_dq AS
SELECT
    c.*,
    filter(array(
      CASE WHEN n.minimum_cause_minutes < 0 THEN 'DQ-CAU-001' END,
      CASE WHEN e.eligibility_failures > 0 THEN 'DQ-CAU-001' END,
      CASE WHEN r.cause_reconciliation_check = 'FAIL' THEN 'DQ-CAU-001' END
    ), x -> x IS NOT NULL) AS failed_rule_ids,

    filter(array(
      CASE WHEN n.minimum_cause_minutes < 0 THEN 'Cause minutes contain a negative value' END,
      CASE WHEN e.eligibility_failures > 0 THEN 'Cause rows exist for an ineligible/not-delayed flight' END,
      CASE WHEN r.cause_reconciliation_check = 'FAIL' THEN 'Cause total does not reconcile to operational delay within tolerance' END
    ), x -> x IS NOT NULL) AS failure_reasons

FROM silver_candidate_delay_components c
LEFT JOIN (
    SELECT flight_id, MIN(cause_minutes) AS minimum_cause_minutes
    FROM silver_candidate_delay_components
    GROUP BY flight_id
) n ON c.flight_id = n.flight_id
LEFT JOIN (
    SELECT flight_id, COUNT(*) AS eligibility_failures
    FROM causes_checked
    WHERE cause_eligibility_check = 'FAIL'
    GROUP BY flight_id
) e ON c.flight_id = e.flight_id
LEFT JOIN causes_reconciled r
  ON c.flight_id = r.flight_id
 AND c.source_record_key = r.source_record_key;

The previous cell retains the physical cause row once while attaching the flight-level reconciliation result. The route decision is still made once per physical cause row.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW causes_routed AS
SELECT
    *,
    CASE WHEN size(failed_rule_ids) = 0 THEN 'PASS' ELSE 'FAIL' END AS dq_status,
    CASE WHEN size(failed_rule_ids) > 0 THEN 'MAJOR' ELSE 'NONE' END AS severity,
    CONCAT_WS('|', flight_id, cause_type) AS physical_record_key,
    _source_file_name AS source_file,
    _ingestion_run_id AS batch_id,
    CASE WHEN size(failed_rule_ids) > 0 THEN current_timestamp() ELSE NULL END AS quarantined_at,
    CAST(NULL AS STRING) AS replay_run_id
FROM causes_dq;

## 15. Route all five Candidate entity sets exactly once

The approved guide requires five Trusted/Quarantine entity pairs. Every Candidate physical record goes to exactly one side.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airports_routed AS
SELECT
    *,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'CRITICAL' ELSE 'NONE' END AS severity,
    airport_code AS physical_record_key,
    _source_file_name AS source_file,
    _ingestion_run_id AS batch_id,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN current_timestamp() ELSE NULL END AS quarantined_at,
    CAST(NULL AS STRING) AS replay_run_id,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN array('DQ-REF-001') ELSE array() END AS failed_rule_ids,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN array('Airport does not resolve to exactly one active reference') ELSE array() END AS failure_reasons,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'airport_code/active_flag' ELSE '' END AS affected_fields
FROM airports_checked;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carriers_routed AS
SELECT
    *,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'CRITICAL' ELSE 'NONE' END AS severity,
    carrier_code AS physical_record_key,
    _source_file_name AS source_file,
    _ingestion_run_id AS batch_id,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN current_timestamp() ELSE NULL END AS quarantined_at,
    CAST(NULL AS STRING) AS replay_run_id,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN array('DQ-REF-001') ELSE array() END AS failed_rule_ids,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN array('Carrier does not resolve to exactly one active reference') ELSE array() END AS failure_reasons,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'carrier_code/active_flag' ELSE '' END AS affected_fields
FROM carriers_checked;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW routes_routed AS
SELECT
    *,
    CASE WHEN dq_ref_001_check = 'PASS' AND dq_rte_001_check = 'PASS' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
    CASE WHEN dq_ref_001_check = 'FAIL' THEN 'CRITICAL'
         WHEN dq_rte_001_check = 'FAIL' THEN 'MAJOR'
         ELSE 'NONE' END AS severity,
    CONCAT_WS('|', route_id, origin_airport_code, destination_airport_code) AS physical_record_key,
    _source_file_name AS source_file,
    _ingestion_run_id AS batch_id,
    CASE WHEN dq_ref_001_check = 'FAIL' OR dq_rte_001_check = 'FAIL'
         THEN current_timestamp() ELSE NULL END AS quarantined_at,
    CAST(NULL AS STRING) AS replay_run_id,
    filter(array(
      CASE WHEN dq_ref_001_check = 'FAIL' THEN 'DQ-REF-001' END,
      CASE WHEN dq_rte_001_check = 'FAIL' THEN 'DQ-RTE-001' END
    ), x -> x IS NOT NULL) AS failed_rule_ids,
    filter(array(
      CASE WHEN dq_ref_001_check = 'FAIL' THEN 'Directed route does not resolve to exactly one approved route' END,
      CASE WHEN dq_rte_001_check = 'FAIL' THEN 'Route identity, direction, distance or band validation failed' END
    ), x -> x IS NOT NULL) AS failure_reasons,
    array_join(filter(array(
      CASE WHEN dq_ref_001_check = 'FAIL' THEN 'origin/destination/route_id' END,
      CASE WHEN dq_rte_001_check = 'FAIL' THEN 'origin/destination/distance/distance_band' END
    ), x -> x IS NOT NULL), '; ') AS affected_fields
FROM routes_checked;

## 16. Write Trusted Silver and Quarantine Delta tables

Gold must read Trusted Silver only. Quarantine retains the original Candidate columns plus DQ/audit metadata.

`CREATE OR REPLACE TABLE` is used here as the controlled snapshot pattern demonstrated by the supplied Week 6 material. The original Candidate tables are not modified.

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_airports USING DELTA AS
SELECT * FROM airports_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_airports USING DELTA AS
SELECT * FROM airports_routed WHERE dq_status = 'FAIL';

CREATE OR REPLACE TABLE trusted_silver_carriers USING DELTA AS
SELECT * FROM carriers_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_carriers USING DELTA AS
SELECT * FROM carriers_routed WHERE dq_status = 'FAIL';

CREATE OR REPLACE TABLE trusted_silver_routes USING DELTA AS
SELECT * FROM routes_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_routes USING DELTA AS
SELECT * FROM routes_routed WHERE dq_status = 'FAIL';

CREATE OR REPLACE TABLE trusted_silver_flights USING DELTA AS
SELECT * FROM flights_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_flights USING DELTA AS
SELECT * FROM flights_routed WHERE dq_status = 'FAIL';

CREATE OR REPLACE TABLE trusted_silver_delay_components USING DELTA AS
SELECT * FROM causes_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_delay_components USING DELTA AS
SELECT * FROM causes_routed WHERE dq_status = 'FAIL';

## 17. Inspect DQ evidence

A passing record must have no failed rule IDs. A quarantined physical record must carry every failed rule ID on the same physical row.

In [ ]:
%sql
SELECT
    flight_id,
    source_record_key,
    cancelled_flag,
    cancellation_code,
    origin_airport_code,
    destination_airport_code,
    dq_status,
    failed_rule_ids,
    failure_reasons,
    severity,
    affected_fields,
    physical_record_key,
    source_file,
    batch_id,
    quarantined_at,
    replay_run_id
FROM quarantine_flights
WHERE size(failed_rule_ids) > 0
LIMIT 20;

In [ ]:
%sql
SELECT
    dq_status,
    COUNT(*) AS rows
FROM trusted_silver_flights
GROUP BY dq_status
UNION ALL
SELECT dq_status, COUNT(*)
FROM quarantine_flights
GROUP BY dq_status;

## 18. Rule scorecard

Rule-failure totals may exceed the number of quarantined rows because one physical record can fail several rules. This is expected and required.

In [ ]:
%sql
SELECT 'DQ-FLT-001' AS rule_id, SUM(CASE WHEN dq_flt_001_check = 'FAIL' THEN 1 ELSE 0 END) AS failed_rows FROM flights_all_checked
UNION ALL
SELECT 'DQ-REF-001', SUM(CASE WHEN dq_ref_001_check = 'FAIL' THEN 1 ELSE 0 END) FROM flights_all_checked
UNION ALL
SELECT 'DQ-TIM-001', SUM(CASE WHEN dq_tim_001_check = 'FAIL' THEN 1 ELSE 0 END) FROM flights_all_checked
UNION ALL
SELECT 'DQ-STS-001', SUM(CASE WHEN dq_sts_001_check = 'FAIL' THEN 1 ELSE 0 END) FROM flights_all_checked
UNION ALL
SELECT 'DQ-DIV-001', SUM(CASE WHEN dq_div_001_check = 'FAIL' THEN 1 ELSE 0 END) FROM flights_all_checked
UNION ALL
SELECT 'DQ-DLY-001', SUM(CASE WHEN dq_dly_001_check = 'FAIL' THEN 1 ELSE 0 END) FROM flights_all_checked
UNION ALL
SELECT 'DQ-RTE-001', SUM(CASE WHEN dq_rte_001_check = 'FAIL' THEN 1 ELSE 0 END) FROM flights_all_checked
UNION ALL
SELECT 'DQ-CAU-001', COUNT(DISTINCT flight_id)
FROM causes_reconciled
WHERE cause_reconciliation_check = 'FAIL'
   OR cause_total_minutes IS NOT NULL AND cause_total_minutes < 0;

## 19. Mandatory no-silent-loss reconciliation

The approved guide requires exact physical-grain reconciliation:

`Candidate distinct = Trusted distinct + Quarantine distinct`

and zero Trusted/Quarantine intersection.

For Flights, the physical key is the Candidate `flight_id`. For Causes, the physical key is `flight_id + cause_type`.

In [ ]:
%sql
SELECT
  'flights' AS entity,
  (SELECT COUNT(DISTINCT flight_id) FROM silver_flights_candidate) AS candidate_distinct,
  (SELECT COUNT(DISTINCT flight_id) FROM trusted_silver_flights) AS trusted_distinct,
  (SELECT COUNT(DISTINCT flight_id) FROM quarantine_flights) AS quarantine_distinct,
  (SELECT COUNT(DISTINCT flight_id) FROM silver_flights_candidate)
    = (SELECT COUNT(DISTINCT flight_id) FROM trusted_silver_flights)
    + (SELECT COUNT(DISTINCT flight_id) FROM quarantine_flights) AS reconciliation_pass
UNION ALL
SELECT
  'delay_components',
  (SELECT COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) FROM silver_candidate_delay_components),
  (SELECT COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) FROM trusted_silver_delay_components),
  (SELECT COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) FROM quarantine_delay_components),
  (SELECT COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) FROM silver_candidate_delay_components)
    = (SELECT COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) FROM trusted_silver_delay_components)
    + (SELECT COUNT(DISTINCT CONCAT_WS('|', flight_id, cause_type)) FROM quarantine_delay_components) AS reconciliation_pass;

In [ ]:
%sql
SELECT COUNT(*) AS flight_trusted_quarantine_intersection
FROM (
    SELECT DISTINCT flight_id FROM trusted_silver_flights
    INTERSECT
    SELECT DISTINCT flight_id FROM quarantine_flights
);

SELECT COUNT(*) AS cause_trusted_quarantine_intersection
FROM (
    SELECT DISTINCT CONCAT_WS('|', flight_id, cause_type) AS physical_key
    FROM trusted_silver_delay_components
    INTERSECT
    SELECT DISTINCT CONCAT_WS('|', flight_id, cause_type) AS physical_key
    FROM quarantine_delay_components
);

### Physical membership proof

The count equality is necessary but does not by itself prove that the same physical records were routed. The following checks that every Candidate physical key occurs exactly once across Trusted + Quarantine.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flight_route_membership AS
SELECT flight_id
FROM trusted_silver_flights
UNION ALL
SELECT flight_id
FROM quarantine_flights;

SELECT COUNT(*) AS candidate_flight_keys_not_routed_once
FROM (
    SELECT c.flight_id
    FROM (SELECT DISTINCT flight_id FROM silver_flights_candidate) c
    LEFT JOIN (
        SELECT flight_id, COUNT(*) AS route_occurrences
        FROM flight_route_membership
        GROUP BY flight_id
    ) r ON c.flight_id = r.flight_id
    WHERE COALESCE(r.route_occurrences, 0) <> 1
) x;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW cause_route_membership AS
SELECT CONCAT_WS('|', flight_id, cause_type) AS physical_key
FROM trusted_silver_delay_components
UNION ALL
SELECT CONCAT_WS('|', flight_id, cause_type)
FROM quarantine_delay_components;

SELECT COUNT(*) AS candidate_cause_keys_not_routed_once
FROM (
    SELECT c.physical_key
    FROM (
        SELECT DISTINCT CONCAT_WS('|', flight_id, cause_type) AS physical_key
        FROM silver_candidate_delay_components
    ) c
    LEFT JOIN (
        SELECT physical_key, COUNT(*) AS route_occurrences
        FROM cause_route_membership
        GROUP BY physical_key
    ) r ON c.physical_key = r.physical_key
    WHERE COALESCE(r.route_occurrences, 0) <> 1
) x;

## 20. Final evidence — trace one multi-failure Flight

The approved Week 06 evidence asks for a Candidate flight with both route and cancellation failures traced into exactly one quarantine row.

First search actual quarantined data. Do not fabricate a defect if the current source does not contain the requested combination.

In [ ]:
%sql
SELECT
    flight_id,
    source_record_key,
    cancelled_flag,
    cancellation_code,
    origin_airport_code,
    destination_airport_code,
    distance_miles,
    dq_status,
    failed_rule_ids,
    failure_reasons,
    severity,
    affected_fields,
    physical_record_key,
    source_file,
    batch_id,
    quarantined_at
FROM quarantine_flights
WHERE array_contains(failed_rule_ids, 'DQ-STS-001')
  AND array_contains(failed_rule_ids, 'DQ-RTE-001')
LIMIT 10;

If the query returns no rows, record that fact. Do not manufacture a multi-failure example. You can still demonstrate the same routing mechanics with a controlled temporary test record in the replay section, but it must be labelled as a test fixture rather than actual project evidence.

## 21. Controlled correction and replay

Required safe pattern:

1. Choose one actual quarantined Flight.
2. Explain every failed rule.
3. Correct the earliest wrong Candidate/source value in a controlled rework batch.
4. Assign a new `replay_run_id`.
5. Rerun **all eight governing checks**, not only the rule that originally failed.
6. Retain the original quarantine evidence.
7. Reconcile again.

Never update Trusted directly, erase the original failure, or rerun only the first failed rule.

In [ ]:
%sql
-- Select one actual quarantined flight for replay.
CREATE OR REPLACE TEMP VIEW replay_source AS
SELECT *
FROM quarantine_flights
WHERE size(failed_rule_ids) > 0
ORDER BY quarantined_at
LIMIT 1;

SELECT
    flight_id,
    failed_rule_ids,
    failure_reasons,
    affected_fields,
    quarantined_at
FROM replay_source;

### Replay fixture

The next cell shows a controlled correction without modifying the original Trusted or Quarantine tables. It corrects the first available failing field for demonstration purposes and assigns a new replay ID.

**Important:** this is a demonstration pattern. For your final evidence, replace the demonstration correction with the actual earliest wrong source/Candidate value identified by your team.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW replay_candidate AS
SELECT
    *,
    'W06-REPLAY-' || date_format(current_timestamp(), 'yyyyMMddHHmmss') AS replay_run_id,
    CASE
      WHEN array_contains(failed_rule_ids, 'DQ-RTE-001')
           AND origin_airport_code <> destination_airport_code
      THEN distance_miles
      WHEN array_contains(failed_rule_ids, 'DQ-STS-001')
           AND cancelled_flag = 1
      THEN NULL
      ELSE distance_miles
    END AS replay_distance_miles,
    CASE
      WHEN array_contains(failed_rule_ids, 'DQ-STS-001')
           AND cancelled_flag = 1
      THEN NULL
      ELSE actual_arrival_hhmm
    END AS replay_actual_arrival_hhmm
FROM replay_source;

SELECT
    flight_id,
    failed_rule_ids AS original_failed_rule_ids,
    replay_run_id,
    replay_distance_miles,
    replay_actual_arrival_hhmm
FROM replay_candidate;

### Replay limitation

The controlled replay above demonstrates the required audit pattern, but the exact corrected value must come from the earliest wrong source/Candidate value established by the team. The original quarantine row remains untouched.

## 22. Controlled rerun proof

Rerunning the DQ notebook should reproduce the same routing logic from the same Candidate snapshot. Re-execute the helper views, checked views, routed views and table-write cells in order, then rerun the reconciliation cells.

Do not type expected counts into the notebook; use actual Databricks results.

In [ ]:
%sql
SELECT
    'flights' AS entity,
    (SELECT COUNT(*) FROM silver_flights_candidate) AS candidate_rows,
    (SELECT COUNT(*) FROM trusted_silver_flights) AS trusted_rows,
    (SELECT COUNT(*) FROM quarantine_flights) AS quarantine_rows,
    CASE WHEN
      (SELECT COUNT(*) FROM silver_flights_candidate)
      =
      (SELECT COUNT(*) FROM trusted_silver_flights)
      +
      (SELECT COUNT(*) FROM quarantine_flights)
    THEN 'PASS' ELSE 'CHECK' END AS status
UNION ALL
SELECT
    'delay_components',
    (SELECT COUNT(*) FROM silver_candidate_delay_components),
    (SELECT COUNT(*) FROM trusted_silver_delay_components),
    (SELECT COUNT(*) FROM quarantine_delay_components),
    CASE WHEN
      (SELECT COUNT(*) FROM silver_candidate_delay_components)
      =
      (SELECT COUNT(*) FROM trusted_silver_delay_components)
      +
      (SELECT COUNT(*) FROM quarantine_delay_components)
    THEN 'PASS' ELSE 'CHECK' END;

## 23. Delta history and audit evidence

In [ ]:
%sql
DESCRIBE HISTORY trusted_silver_flights LIMIT 5;

DESCRIBE HISTORY quarantine_flights LIMIT 5;

DESCRIBE HISTORY trusted_silver_delay_components LIMIT 5;

DESCRIBE HISTORY quarantine_delay_components LIMIT 5;

## 24. Week 06 summary

The Week 06 pipeline is now:

```text
Week-5 Silver Candidate
        ↓
Reference master validation
        ↓
Eight governed DQ rules
        ↓
Independent PASS/FAIL evaluation
        ↓
One physical row → exactly one destination
        ├── Trusted Silver
        └── Quarantine
        ↓
Exact reconciliation + zero intersection
        ↓
Controlled correction + new replay_run_id
        ↓
Full DQ replay
```

### Required outputs

- `trusted_silver_airports`
- `quarantine_airports`
- `trusted_silver_carriers`
- `quarantine_carriers`
- `trusted_silver_routes`
- `quarantine_routes`
- `trusted_silver_flights`
- `quarantine_flights`
- `trusted_silver_delay_components`
- `quarantine_delay_components`

### Required DQ metadata

- `dq_status`
- `failed_rule_ids`
- `failure_reasons`
- `severity`
- `affected_fields`
- `physical_record_key`
- `source_file`
- `batch_id`
- `quarantined_at`
- `replay_run_id`

### Week 06 acceptance checklist

- [ ] All eight exact rule IDs and severities are implemented.
- [ ] Reference masters are validated before Flight checks.
- [ ] Signed and operational delay meanings remain separate.
- [ ] Cancellation/diversion logic respects allowed NULLs and `U`.
- [ ] All applicable failures are retained; no short-circuit logic.
- [ ] One physical quarantine row carries all failed IDs.
- [ ] Five Trusted/Quarantine entity pairs exist.
- [ ] Flight and cause reconciliations pass exactly.
- [ ] Trusted/Quarantine intersection is zero.
- [ ] One corrected record re-enters Candidate and is subjected to full DQ.
- [ ] Notebook, rule source, summary and Week 06 log are updated.

### Submission evidence to capture

1. Candidate handoff and starting counts.
2. Rule scorecard.
3. One multi-failure Flight quarantine row, if the actual data contains one.
4. Flight and cause reconciliation.
5. Zero Trusted/Quarantine intersection.
6. Controlled replay with a new `replay_run_id`.
7. Delta history for Trusted and Quarantine tables.

## AI Transparency Note

This notebook was generated by comparing the supplied Week-5 SkyOps notebook, Week-6 learning material, project data dictionary/DQ requirements, and the approved Team 07 Week-06 DQ guide.

Where the supplied materials did not provide a numeric cause tolerance or fixed numeric distance-band thresholds, the notebook makes that gap explicit and exposes configuration rather than silently presenting an invented project rule.

Before submission, execute every SQL cell in the team's Databricks workspace and verify the actual counts, schemas, rule failures, reconciliations and replay evidence.